# M2-B2 — Anonymisation Franck

Objectif de cette étape : charger `audit_sample.csv` (200 lignes stratifiées sur `income`) et analyser les types de PII présents dans `manager_comments` (noms, emails, téléphones, IBAN partiels, dates, lieux).

In [1]:
from pathlib import Path
import re

import pandas as pd

RANDOM_STATE = 42
DATA_DIR = Path("../data")
SAMPLE_PATH = DATA_DIR / "audit_sample.csv"

df = pd.read_csv(SAMPLE_PATH)
comments = df["manager_comments"].fillna("").astype(str)

display(pd.DataFrame({"rows": [len(df)], "columns": [df.shape[1]]}))
display(df[["income", "manager_comments"]].head(5))

,rows,columns
0,200,16


,income,manager_comments
0,<=50K,Daniel Murphy flagged a conflict with Gregory ...
1,<=50K,Dennis Huff requested a transfer. Approved by ...
2,<=50K,Strong year for Cynthia Hardy. Bonus wired to ...
3,>50K,Plan d'accompagnement ouvert pour Pierre Lapor...
4,>50K,Strong year for Matthew Acevedo. Bonus wired t...


In [4]:
# Verification de la stratification sur income : full dataset vs sample
FULL_PATH = DATA_DIR / "adult_income_with_comments.csv"
df_full = pd.read_csv(FULL_PATH)

income_compare = pd.concat(
    [
        df_full["income"].value_counts(normalize=True).rename("full_share"),
        df["income"].value_counts(normalize=True).rename("sample_share"),
    ],
    axis=1,
).fillna(0.0)

income_compare["abs_diff_pct_points"] = ((income_compare["sample_share"] - income_compare["full_share"]).abs() * 100).round(2)
income_compare = income_compare.assign(
    full_share=lambda d: (100 * d["full_share"]).round(2),
    sample_share=lambda d: (100 * d["sample_share"]).round(2),
)

display(income_compare)
display(pd.DataFrame({"max_abs_diff_pct_points": [income_compare["abs_diff_pct_points"].max()]}))

,full_share,sample_share,abs_diff_pct_points
income,,,
<=50K,75.92,50.0,25.92
>50K,24.08,50.0,25.92


,max_abs_diff_pct_points
0,25.92


In [2]:
# Regex heuristiques pour identifier les PII dans les commentaires
pattern_map = {
    "name": re.compile(r"\b[A-Z][a-z]+(?:[-'][A-Z][a-z]+)?\s[A-Z][a-z]+(?:[-'][A-Z][a-z]+)?\b"),
    "email": re.compile(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b"),
    "phone": re.compile(r"(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{2,4}\)?[\s.-]?)\d{2,4}[\s.-]?\d{2,4}[\s.-]?\d{2,4}"),
    "iban_partial": re.compile(r"\b[A-Z]{2}\d{2}(?:[ -]?[A-Z0-9]{2,4}){2,}\b"),
    "date": re.compile(r"\b(?:\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|\d{4}-\d{2}-\d{2})\b"),
}

# Lieux : on exploite les valeurs de native_country quand elles apparaissent dans les commentaires
countries = sorted(
    {str(c).strip() for c in df["native_country"].dropna().unique() if str(c).strip()},
    key=len,
    reverse=True,
 )
country_pattern = re.compile("|".join(re.escape(c) for c in countries)) if countries else None

records = []
for pii_type, pattern in pattern_map.items():
    row_count = comments.apply(lambda txt: bool(pattern.search(txt))).sum()
    total_matches = comments.apply(lambda txt: len(pattern.findall(txt))).sum()
    records.append({
        "pii_type": pii_type,
        "rows_with_pii": int(row_count),
        "total_matches": int(total_matches),
        "share_rows_pct": round(100 * row_count / len(comments), 2),
    })

if country_pattern is not None:
    location_rows = comments.apply(lambda txt: bool(country_pattern.search(txt))).sum()
    location_matches = comments.apply(lambda txt: len(country_pattern.findall(txt))).sum()
else:
    location_rows = 0
    location_matches = 0

records.append(
    {
        "pii_type": "location",
        "rows_with_pii": int(location_rows),
        "total_matches": int(location_matches),
        "share_rows_pct": round(100 * location_rows / len(comments), 2),
    }
)

pii_frequency = pd.DataFrame(records).sort_values("rows_with_pii", ascending=False).reset_index(drop=True)
display(pii_frequency)

,pii_type,rows_with_pii,total_matches,share_rows_pct
0,name,198,384,99.0
1,phone,119,146,59.5
2,date,86,86,43.0
3,email,85,85,42.5
4,iban_partial,0,0,0.0
5,location,0,0,0.0


In [5]:
# Exemples pour inspection qualitative (2 exemples par type)
def extract_examples(text_series: pd.Series, regex: re.Pattern, max_examples: int = 2) -> pd.DataFrame:
    matched = text_series[text_series.apply(lambda txt: bool(regex.search(txt)))].head(max_examples)
    if matched.empty:
        return pd.DataFrame({"example": ["Aucun exemple trouvé"]})
    return pd.DataFrame({"example": matched.values})

def show_examples(df_examples: pd.DataFrame) -> None:
    styled = df_examples.style.set_properties(
        subset=["example"],
        **{"white-space": "pre-wrap", "text-align": "left"},
    )
    display(styled)

with pd.option_context("display.max_colwidth", None):
    for pii_type, pattern in pattern_map.items():
        display(pd.DataFrame({"pii_type": [pii_type]}))
        show_examples(extract_examples(comments, pattern, max_examples=2))

    display(pd.DataFrame({"pii_type": ["location"]}))
    if country_pattern is not None:
        show_examples(extract_examples(comments, country_pattern, max_examples=2))
    else:
        show_examples(pd.DataFrame({"example": ["Aucun pays exploitable dans native_country"]}))

,pii_type
0,name


,example
0,Daniel Murphy flagged a conflict with Gregory Meyer. Mediation scheduled. HR follow-up: ashleyraymond@example.org.
1,Dennis Huff requested a transfer. Approved by Courtney Miller. Reach them at 310.727.4959 for handover. Ticket HR-24680.


,pii_type
0,email


,example
0,Daniel Murphy flagged a conflict with Gregory Meyer. Mediation scheduled. HR follow-up: ashleyraymond@example.org.
1,Plan d'accompagnement ouvert pour Pierre Laporte. Coaching confié à Pauline Monnier (rcordova@example.net).


,pii_type
0,phone


...

,example
0,Aucun exemple trouvé


,pii_type
0,date


,example
0,Strong year for Cynthia Hardy. Bonus wired to account ****1400. Validated by Brittany Garcia on 2025-02-11.
1,Strong year for Matthew Acevedo. Bonus wired to account ****8655. Validated by David Parrish on 2025-08-03.


,pii_type
0,location


,example
0,Aucun exemple trouvé
